
# DJI MRK → ASCII PLY Converter (CMOS → Antenna correction + quality colors)

This notebook reads a **DJI RTK `.MRK`** file, converts the positions to a metric CRS (default **Belgian Lambert 2008 / EPSG:3812**), and exports an **ASCII PLY** point cloud.

## What it does
- Parses DJI MRK fields:
  - `Lat`, `Lon`, `Ellh`
  - `N`, `E`, `V` (camera-to-antenna offset in **mm**)
  - `Q` signal quality code
- Converts coordinates from **WGS84 3D** (`EPSG:4979`) to your target metric CRS
- Applies a **CMOS sensor → antenna** correction using the MRK `N/E/V` offsets
  - done in a proper local ENU frame (via ECEF), then projected
- Colors points by signal quality:
  - **bad quality = red**
  - **otherwise = orange**
- Exports:
  - **ASCII PLY** (`x y z r g b`)
  - optional **TXT** sidecar with parsed + transformed values (default = True)

## Default assumptions
- `N/E/V` in the MRK file are interpreted as the vector from **CMOS sensor → antenna**
- `V` is **up-positive** (as you noted)
- By default, `Q >= 50` is treated as **good** (orange), lower values as **bad** (red)

You can change all defaults in the **Config** cell.


In [ ]:

# If needed (run once):
# !pip install pyproj pandas numpy tqdm ipywidgets

import os
import re
import math
import warnings
from pathlib import Path
from typing import Optional, Tuple, Dict, Any

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

from pyproj import CRS, Transformer
from pyproj import datadir as pyproj_datadir
from pyproj.transformer import TransformerGroup

from IPython.display import display

print("Imports OK")


#### Config

In [ ]:

# =========================
# Config (all hard-written paths + declarations here)
# =========================

# Default suggested MRK file (edit if needed)
INPUT_MRK_PATH = r"C:\Users\EB\Desktop\ORBIT2.0\Orbit2\4_RawData\2_ATR\3_RTK\DJI_202602201316_684_Spoortest-05MerendreebrugUnderdeckFlightRouteSection2_Timestamp.MRK"

# Source and target CRS
# MRK Lat/Lon/Ellh is geographic WGS84 3D -> EPSG:4979
INPUT_CRS = "EPSG:4979"

# Metric output CRS (Belgium)
# Recommended: EPSG:3812 (Belgian Lambert 2008)
# Alternative:  EPSG:31370 (Belgian Lambert 72)
OUTPUT_CRS = "EPSG:3812"

# Optional local grid folder (same default as your earlier tool)
# Can contain e.g. _BD72LB72_ETRS89LB08.gsb and hBG18.geo
GRID_DIR = r"C:\Code\_Orbitv1.0.4\orbit\tools\CSV2PLY"

# Apply CMOS sensor -> antenna shift using MRK N/E/V offsets
APPLY_CMOS_TO_ANTENNA_SHIFT = True

# Export an extra PLY with the unshifted CMOS points (for comparison)
EXPORT_CMOS_PLY_TOO = False

# Signal quality handling
# Default rule: Q >= 50 = good (orange), lower = bad (red)
Q_GOOD_MIN = 50

# Colors (RGB)
COLOR_BAD = (255, 0, 0)       # red
COLOR_OK  = (255, 165, 0)     # orange

# Sidecar TXT export with all parsed/transformed columns
EXPORT_TXT = True
TXT_DELIMITER = "\t"

# Output formatting
DECIMALS = 6
OVERWRITE = True

# Optional console printing of point quality rows (can be long for large files)
PRINT_ALL_POINT_QUALITY_ROWS = False
PRINT_FIRST_N_POINT_QUALITY_ROWS = 20   # used if PRINT_ALL_POINT_QUALITY_ROWS = False


In [ ]:

# =========================
# Core helpers
# =========================

def _configure_proj_grids(grid_dir: Optional[str]) -> bool:
    if not grid_dir:
        return False
    p = Path(grid_dir)
    if not p.exists():
        warnings.warn(f"Grid directory does not exist: {p}")
        return False
    try:
        pyproj_datadir.append_data_dir(str(p))
    except Exception:
        pass
    os.environ.setdefault("PROJ_NETWORK", "OFF")
    return True


def _parse_tagged_numeric(token: str, tag: str) -> float:
    # Examples:
    #   "   -46,N"  -> -46
    #   "51.07370078,Lat" -> 51.07370078
    s = str(token).strip()
    m = re.match(r"^\s*([+-]?\d+(?:\.\d+)?)\s*,\s*" + re.escape(tag) + r"\s*$", s, flags=re.I)
    if m:
        return float(m.group(1))
    if s.lower().endswith("," + tag.lower()):
        return float(s[:-(len(tag)+1)].strip())
    raise ValueError(f"Could not parse token '{token}' as '<number>,{tag}'")


def read_dji_mrk(mrk_path: str) -> pd.DataFrame:
    rows = []
    malformed = []

    with open(mrk_path, "r", encoding="utf-8", errors="ignore") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue

            # DJI MRK is tab-separated, but spacing can vary inside tokens
            parts = [p for p in line.rstrip("\n").split("\t") if p.strip() != ""]

            try:
                if len(parts) < 11:
                    raise ValueError(f"Expected >=11 fields, got {len(parts)}")

                row_idx = int(parts[0].strip())
                gps_sow = float(parts[1].strip())
                gps_week = int(parts[2].strip().strip("[]"))

                n_mm = _parse_tagged_numeric(parts[3], "N")
                e_mm = _parse_tagged_numeric(parts[4], "E")
                v_mm = _parse_tagged_numeric(parts[5], "V")

                lat = _parse_tagged_numeric(parts[6], "Lat")
                lon = _parse_tagged_numeric(parts[7], "Lon")
                ellh = _parse_tagged_numeric(parts[8], "Ellh")

                # Example field: "0.014073, 0.011575, 0.025236"
                sigma_chunks = [x.strip() for x in parts[9].split(",")]
                if len(sigma_chunks) >= 3:
                    sigma_1 = float(sigma_chunks[0])
                    sigma_2 = float(sigma_chunks[1])
                    sigma_3 = float(sigma_chunks[2])
                else:
                    sigma_1 = sigma_2 = sigma_3 = np.nan

                q_code = int(_parse_tagged_numeric(parts[10], "Q"))

                rows.append({
                    "line_no": line_no,
                    "index": row_idx,
                    "gps_sow": gps_sow,
                    "gps_week": gps_week,
                    "n_mm": n_mm,
                    "e_mm": e_mm,
                    "v_mm": v_mm,
                    "lat": lat,
                    "lon": lon,
                    "ellh": ellh,
                    "sigma_1": sigma_1,
                    "sigma_2": sigma_2,
                    "sigma_3": sigma_3,
                    "q_code": q_code,
                    "raw_line": line.rstrip("\n"),
                })

            except Exception as e:
                malformed.append((line_no, str(e), line[:180].rstrip()))

    if not rows:
        raise RuntimeError("No valid MRK rows were parsed.")

    df = pd.DataFrame(rows)

    if malformed:
        print(f"Warning: skipped {len(malformed)} malformed line(s).")
        for line_no, msg, raw in malformed[:5]:
            print(f"  line {line_no}: {msg} | {raw}")

    return df


def _build_transformer(input_crs: str, output_crs: str, grid_dir: Optional[str] = None):
    _configure_proj_grids(grid_dir)

    src = CRS.from_user_input(input_crs)
    dst = CRS.from_user_input(output_crs)

    tg = TransformerGroup(src, dst, always_xy=True)
    if not tg.transformers:
        raise RuntimeError(f"No transformation available from {src} to {dst}")

    transformer = tg.transformers[0]
    return transformer, src, dst, tg


def _enu_shift_geodetic_arrays(
    lon_deg: np.ndarray,
    lat_deg: np.ndarray,
    h_m: np.ndarray,
    e_m: np.ndarray,
    n_m: np.ndarray,
    u_m: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Apply a local ENU offset (meters) to geodetic WGS84 coordinates using ECEF.
    Returns shifted lon, lat, h (WGS84 / EPSG:4979).
    """
    lon_deg = np.asarray(lon_deg, dtype=float)
    lat_deg = np.asarray(lat_deg, dtype=float)
    h_m = np.asarray(h_m, dtype=float)
    e_m = np.asarray(e_m, dtype=float)
    n_m = np.asarray(n_m, dtype=float)
    u_m = np.asarray(u_m, dtype=float)

    geod_to_ecef = Transformer.from_crs("EPSG:4979", "EPSG:4978", always_xy=True)
    ecef_to_geod = Transformer.from_crs("EPSG:4978", "EPSG:4979", always_xy=True)

    X, Y, Z = geod_to_ecef.transform(lon_deg, lat_deg, h_m)

    lon = np.deg2rad(lon_deg)
    lat = np.deg2rad(lat_deg)

    # ENU basis vectors in ECEF
    east_x  = -np.sin(lon)
    east_y  =  np.cos(lon)
    east_z  =  np.zeros_like(lon)

    north_x = -np.sin(lat) * np.cos(lon)
    north_y = -np.sin(lat) * np.sin(lon)
    north_z =  np.cos(lat)

    up_x    =  np.cos(lat) * np.cos(lon)
    up_y    =  np.cos(lat) * np.sin(lon)
    up_z    =  np.sin(lat)

    dX = e_m * east_x + n_m * north_x + u_m * up_x
    dY = e_m * east_y + n_m * north_y + u_m * up_y
    dZ = e_m * east_z + n_m * north_z + u_m * up_z

    X2 = X + dX
    Y2 = Y + dY
    Z2 = Z + dZ

    lon2, lat2, h2 = ecef_to_geod.transform(X2, Y2, Z2)
    return lon2, lat2, h2


def _quality_to_rgb(q_codes: np.ndarray, q_good_min: int, color_bad=(255,0,0), color_ok=(255,165,0)):
    q = np.asarray(q_codes, dtype=int)
    is_bad = q < int(q_good_min)

    rb, gb, bb = color_bad
    ro, go, bo = color_ok

    red   = np.where(is_bad, rb, ro).astype(np.uint8)
    green = np.where(is_bad, gb, go).astype(np.uint8)
    blue  = np.where(is_bad, bb, bo).astype(np.uint8)

    label = np.where(is_bad, "bad", "ok")
    return is_bad, red, green, blue, label


def _write_ascii_ply_xyzrgb(out_ply: Path, xyz: np.ndarray, rgb: np.ndarray, decimals: int = 6):
    out_ply = Path(out_ply)
    out_ply.parent.mkdir(parents=True, exist_ok=True)

    if xyz.shape[0] != rgb.shape[0]:
        raise ValueError("xyz and rgb row counts do not match")

    n = xyz.shape[0]
    fmt = f"{{:.{decimals}f}} {{:.{decimals}f}} {{:.{decimals}f}} {{}} {{}} {{}}\n"

    with open(out_ply, "w", encoding="utf-8") as f:
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write(f"element vertex {n}\n")
        f.write("property float x\n")
        f.write("property float y\n")
        f.write("property float z\n")
        f.write("property uchar red\n")
        f.write("property uchar green\n")
        f.write("property uchar blue\n")
        f.write("end_header\n")

        for (x, y, z), (r, g, b) in zip(xyz, rgb):
            f.write(fmt.format(float(x), float(y), float(z), int(r), int(g), int(b)))

    return out_ply


def _ansi_color(text: str, rgb: Tuple[int,int,int]) -> str:
    # Notebook terminals usually support ANSI, but if not, it'll just show the raw text
    r, g, b = rgb
    return f"\x1b[38;2;{r};{g};{b}m{text}\x1b[0m"


In [ ]:

# =========================
# Main processing function
# =========================

def process_dji_mrk_to_ply(
    mrk_path: str,
    input_crs: str = "EPSG:4979",
    output_crs: str = "EPSG:3812",
    grid_dir: Optional[str] = None,
    apply_cmos_to_antenna_shift: bool = True,
    export_cmos_ply_too: bool = False,
    q_good_min: int = 50,
    color_bad: Tuple[int,int,int] = (255, 0, 0),
    color_ok: Tuple[int,int,int] = (255, 165, 0),
    export_txt: bool = True,
    txt_delimiter: str = "\t",
    decimals: int = 6,
    overwrite: bool = True,
):
    mrk_path = Path(mrk_path)
    if not mrk_path.exists():
        raise FileNotFoundError(f"MRK file not found: {mrk_path}")

    # Output folders
    out_root = mrk_path.parent
    ply_dir = out_root / "PLY"
    txt_dir = out_root / "TXT"

    # Parse MRK
    df = read_dji_mrk(str(mrk_path))

    # Convert shifts from mm -> m
    df["shift_n_m"] = df["n_mm"] / 1000.0
    df["shift_e_m"] = df["e_mm"] / 1000.0
    df["shift_v_m"] = df["v_mm"] / 1000.0

    # Compute antenna geodetic coordinates from CMOS geodetic + local ENU shift
    lon_ant, lat_ant, ellh_ant = _enu_shift_geodetic_arrays(
        lon_deg=df["lon"].to_numpy(),
        lat_deg=df["lat"].to_numpy(),
        h_m=df["ellh"].to_numpy(),
        e_m=df["shift_e_m"].to_numpy(),
        n_m=df["shift_n_m"].to_numpy(),
        u_m=df["shift_v_m"].to_numpy(),
    )
    df["lon_ant"] = lon_ant
    df["lat_ant"] = lat_ant
    df["ellh_ant"] = ellh_ant

    # Build transformer to metric output CRS
    transformer, src_crs, dst_crs, tg = _build_transformer(
        input_crs=input_crs,
        output_crs=output_crs,
        grid_dir=grid_dir,
    )

    # Project CMOS points
    x_cmos, y_cmos, z_cmos = transformer.transform(
        df["lon"].to_numpy(),
        df["lat"].to_numpy(),
        df["ellh"].to_numpy()
    )
    df["x_cmos"] = x_cmos
    df["y_cmos"] = y_cmos
    df["z_cmos"] = z_cmos

    # Project shifted antenna points
    x_ant, y_ant, z_ant = transformer.transform(
        df["lon_ant"].to_numpy(),
        df["lat_ant"].to_numpy(),
        df["ellh_ant"].to_numpy()
    )
    df["x_ant"] = x_ant
    df["y_ant"] = y_ant
    df["z_ant"] = z_ant

    # Projected delta (useful sanity check)
    df["dx_proj_m"] = df["x_ant"] - df["x_cmos"]
    df["dy_proj_m"] = df["y_ant"] - df["y_cmos"]
    df["dz_proj_m"] = df["z_ant"] - df["z_cmos"]

    # Signal quality -> colors
    is_bad, red, green, blue, q_label = _quality_to_rgb(
        df["q_code"].to_numpy(),
        q_good_min=q_good_min,
        color_bad=color_bad,
        color_ok=color_ok,
    )
    df["is_bad_quality"] = is_bad
    df["q_label"] = q_label
    df["red"] = red
    df["green"] = green
    df["blue"] = blue

    # Pick export coordinates
    coord_prefix = "ant" if apply_cmos_to_antenna_shift else "cmos"
    xyz = np.column_stack([
        df[f"x_{coord_prefix}"].to_numpy(),
        df[f"y_{coord_prefix}"].to_numpy(),
        df[f"z_{coord_prefix}"].to_numpy(),
    ])
    rgb = np.column_stack([
        df["red"].to_numpy(),
        df["green"].to_numpy(),
        df["blue"].to_numpy(),
    ])

    out_stem = (
        f"{mrk_path.stem}_"
        f"{output_crs.replace(':', '-')}_"
        f"{'AntennaShift' if apply_cmos_to_antenna_shift else 'CMOS'}_QColored"
    )

    out_ply = ply_dir / f"{out_stem}.ply"
    if out_ply.exists() and not overwrite:
        raise FileExistsError(f"Output PLY exists and OVERWRITE=False: {out_ply}")

    _write_ascii_ply_xyzrgb(out_ply, xyz=xyz, rgb=rgb, decimals=decimals)

    out_ply_cmos = None
    if export_cmos_ply_too and apply_cmos_to_antenna_shift:
        out_ply_cmos = ply_dir / f"{mrk_path.stem}_{output_crs.replace(':', '-')}_CMOS_QColored.ply"
        if out_ply_cmos.exists() and not overwrite:
            raise FileExistsError(f"Output CMOS PLY exists and OVERWRITE=False: {out_ply_cmos}")
        xyz_cmos = np.column_stack([df["x_cmos"], df["y_cmos"], df["z_cmos"]])
        _write_ascii_ply_xyzrgb(out_ply_cmos, xyz=xyz_cmos, rgb=rgb, decimals=decimals)

    out_txt = None
    if export_txt:
        out_txt = txt_dir / f"{out_stem}.txt"
        out_txt.parent.mkdir(parents=True, exist_ok=True)
        if out_txt.exists() and not overwrite:
            raise FileExistsError(f"Output TXT exists and OVERWRITE=False: {out_txt}")
        df.to_csv(out_txt, sep=txt_delimiter, index=False)

    # Quality summary
    q_summary = (
        df.groupby("q_code", dropna=False)
          .agg(
              n=("q_code", "size"),
              pct=("q_code", lambda s: 100.0 * len(s) / len(df)),
              mean_sigma1_m=("sigma_1", "mean"),
              mean_sigma2_m=("sigma_2", "mean"),
              mean_sigma3_m=("sigma_3", "mean"),
          )
          .reset_index()
          .sort_values("q_code", ascending=False)
    )
    q_summary["class"] = np.where(q_summary["q_code"] < int(q_good_min), "bad (red)", "ok (orange)")
    q_summary = q_summary[["q_code", "class", "n", "pct", "mean_sigma1_m", "mean_sigma2_m", "mean_sigma3_m"]]

    # Short metadata summary
    info = {
        "mrk_path": str(mrk_path),
        "n_points": int(len(df)),
        "input_crs": str(src_crs),
        "output_crs": str(dst_crs),
        "transformer": transformer.description,
        "q_good_min_rule": f"Q >= {q_good_min} => orange; else red",
        "out_ply": str(out_ply),
        "out_txt": str(out_txt) if out_txt else None,
        "out_ply_cmos": str(out_ply_cmos) if out_ply_cmos else None,
    }

    return df, q_summary, info


#### CLI

In [ ]:

# =========================
# CLI / Run cell (uses the Config values above)
# =========================

if not INPUT_MRK_PATH:
    print("Set INPUT_MRK_PATH in the Config cell and run again.")
else:
    df_out, q_summary, run_info = process_dji_mrk_to_ply(
        mrk_path=INPUT_MRK_PATH,
        input_crs=INPUT_CRS,
        output_crs=OUTPUT_CRS,
        grid_dir=GRID_DIR,
        apply_cmos_to_antenna_shift=APPLY_CMOS_TO_ANTENNA_SHIFT,
        export_cmos_ply_too=EXPORT_CMOS_PLY_TOO,
        q_good_min=Q_GOOD_MIN,
        color_bad=COLOR_BAD,
        color_ok=COLOR_OK,
        export_txt=EXPORT_TXT,
        txt_delimiter=TXT_DELIMITER,
        decimals=DECIMALS,
        overwrite=OVERWRITE,
    )

    print("Done.\n")
    print("Signal quality summary (Q code):")
    display(q_summary)

    print("\nRun info:")
    for k, v in run_info.items():
        print(f"- {k}: {v}")

    # Quick sanity check: projected CMOS->antenna correction statistics
    print("\nProjected shift statistics (m) [CMOS -> antenna]:")
    display(df_out[["dx_proj_m", "dy_proj_m", "dz_proj_m"]].describe())

    # Show a preview of colored point-quality rows
    n_preview = len(df_out) if PRINT_ALL_POINT_QUALITY_ROWS else min(PRINT_FIRST_N_POINT_QUALITY_ROWS, len(df_out))
    preview_cols = ["index", "gps_sow", "q_code", "q_label", "lat", "lon", "ellh", "x_ant", "y_ant", "z_ant"]
    print(f"\nPer-point quality preview ({n_preview} row(s)):")
    for _, row in df_out.head(n_preview)[preview_cols + ["is_bad_quality"]].iterrows():
        txt = (
            f"idx={int(row['index']):4d} | sow={row['gps_sow']:.3f} | Q={int(row['q_code']):2d} ({row['q_label']}) | "
            f"lat={row['lat']:.8f}, lon={row['lon']:.8f}, ellh={row['ellh']:.3f}"
        )
        rgb = COLOR_BAD if bool(row['is_bad_quality']) else COLOR_OK
        print(_ansi_color(txt, rgb))

    if PRINT_ALL_POINT_QUALITY_ROWS and len(df_out) > n_preview:
        print(f"... printed all {len(df_out)} rows.")
    elif not PRINT_ALL_POINT_QUALITY_ROWS and len(df_out) > n_preview:
        print(f"... ({len(df_out)-n_preview} more rows not shown; set PRINT_ALL_POINT_QUALITY_ROWS=True to print all)")


In [ ]:

# =========================
# Optional mini UI (simple notebook controls)
# =========================

try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    w_mrk = widgets.Text(
        value=INPUT_MRK_PATH,
        description="MRK:",
        layout=widgets.Layout(width="95%"),
    )
    w_in = widgets.Text(
        value=INPUT_CRS,
        description="Input CRS:",
        layout=widgets.Layout(width="47%"),
    )
    w_out = widgets.Text(
        value=OUTPUT_CRS,
        description="Output CRS:",
        layout=widgets.Layout(width="47%"),
    )
    w_grid = widgets.Text(
        value=GRID_DIR if GRID_DIR else "",
        description="Grid dir:",
        layout=widgets.Layout(width="95%"),
    )
    w_shift = widgets.Checkbox(value=APPLY_CMOS_TO_ANTENNA_SHIFT, description="Apply CMOS→antenna shift")
    w_extra = widgets.Checkbox(value=EXPORT_CMOS_PLY_TOO, description="Export extra CMOS PLY")
    w_txt = widgets.Checkbox(value=EXPORT_TXT, description="Export TXT")
    w_qmin = widgets.IntText(value=Q_GOOD_MIN, description="Q good min:")
    w_run = widgets.Button(description="Process MRK", button_style="success")
    w_out_box = widgets.Output()

    def _run_ui(_):
        with w_out_box:
            clear_output()
            try:
                df_u, q_u, info_u = process_dji_mrk_to_ply(
                    mrk_path=w_mrk.value.strip(),
                    input_crs=w_in.value.strip() or "EPSG:4979",
                    output_crs=w_out.value.strip() or "EPSG:3812",
                    grid_dir=w_grid.value.strip() or None,
                    apply_cmos_to_antenna_shift=w_shift.value,
                    export_cmos_ply_too=w_extra.value,
                    q_good_min=int(w_qmin.value),
                    color_bad=COLOR_BAD,
                    color_ok=COLOR_OK,
                    export_txt=w_txt.value,
                    txt_delimiter=TXT_DELIMITER,
                    decimals=DECIMALS,
                    overwrite=OVERWRITE,
                )
                print("Done.")
                display(q_u)
                for k, v in info_u.items():
                    print(f"- {k}: {v}")
            except Exception as e:
                print("ERROR:", e)

    w_run.on_click(_run_ui)

    display(widgets.VBox([
        w_mrk,
        widgets.HBox([w_in, w_out]),
        w_grid,
        widgets.HBox([w_shift, w_extra, w_txt]),
        widgets.HBox([w_qmin, w_run]),
        w_out_box,
    ]))

except Exception as e:
    print("ipywidgets UI not available:", e)
